# 03 – Fra piksler til prediksjoner

**Mål:** følge formene fra bilder til klassescores,
forstå vektede summer og bygge en lineær modell.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


In [2]:
from pytorch_101.utils import download_fashion_mnist

train_data, test_data = download_fashion_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


Treningsbilder: 20000, testbilder: 1000


## Eksempel: en vektet sum

En lineær modell multipliserer hver input med en vekt, summerer og
legger til et konstantledd (*bias*). Vektene bestemmer påvirkningen fra hver verdi.


Den vektede summen skrives som `z = x @ W + b`, der `@` er matrisemultiplikasjon, `x` input, `W` vekter, og `b` bias.


In [3]:
# To målinger gir én score.
x = torch.tensor([[2.0, 3.0]])
weights = torch.tensor([[0.5], [-1.0]])
bias = torch.tensor([0.2])
score = x @ weights + bias
print("Score:", score, "Form:", score.shape)
print("Samme beregning for hånd:", 2 * 0.5 + 3 * (-1) + 0.2)


Score: tensor([[-1.8000]]) Form: torch.Size([1, 1])
Samme beregning for hånd: -1.8


## Oppgave 1: Endre én vekt og forutsi virkningen

La
`inputs = [[1., 2., 3.], [4., 5., 6.]]`,
`weights = [[1., 0.], [0., 1.], [1., -1.]]`, `bias = [0.5, -0.5]`.
Definer tensorene og beregn `scores`. Øk så vekten i rad 0, kolonne 0 med 1
og beregn `changed_scores`. Hvilke utganger endres, og med hvor mye?
Skriv forventningen i svarcellen før kjøring.

<details><summary>Hint</summary>

Kopier vektene med `clone()`. Vektens rad velger inngang, kolonnen velger utgang.

</details>


In [4]:
inputs = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
weights = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]])
bias = torch.tensor([0.5, -0.5])
scores = inputs @ weights + bias
print(scores, scores.shape)

changed_weights = weights.clone()
changed_weights[0, 0] += 1
changed_scores = inputs @ changed_weights + bias
print("Endring:", changed_scores - scores)


tensor([[ 4.5000, -1.5000],
        [10.5000, -1.5000]]) torch.Size([2, 2])
Endring: tensor([[1., 0.],
        [4., 0.]])


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [5]:
assert torch.allclose(scores, torch.tensor([[4.5, -1.5], [10.5, -1.5]]))

assert torch.allclose(changed_scores - scores, torch.tensor([[1.0, 0.0], [4.0, 0.0]]))


### Refleksjon

Hva representerer radene og kolonnene i resultatet? Hvorfor må de to indre
dimensjonene stemme ved matrisemultiplikasjon? Hvordan ville `*` oppført seg her?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.

Hvilken klassekolonne endret seg? Knytt størrelsen på endringen til inngangsverdiene.


**Mitt svar:**

Hver rad er ett eksempel, og hver kolonne er én score. Vi summerer over de tre
inngangsverdiene, derfor må denne dimensjonen stemme. `*` prøver elementvis
gange; her er formene inkompatible og operasjonen gir en feil.

Bare første utgang endres. Den øker med 1 i første eksempel og 4 i andre,
fordi den første inngangen er henholdsvis 1 og 4. En vekt deles av alle eksemplene.


## Eksempel: flat ut bilder, behold bildeskillet

En lineær modell trenger én rekke tall per bilde. `flatten(start_dim=1)`
beholder dimensjon 0 og slår sammen resten. Piksler fjernes ikke, men den
todimensjonale plasseringen er ikke lenger eksplisitt i formen.


In [6]:
small_images = torch.arange(24, dtype=torch.float32).reshape(2, 1, 3, 4)
flattened = small_images.flatten(start_dim=1)
print(small_images.shape, "→", flattened.shape)


torch.Size([2, 1, 3, 4]) → torch.Size([2, 12])


## DataLoader

Til nå har vi hentet enkeltbilder med `train_data[index]`. PyTorch kan (og bør) regne på flere bilder samtidig (i parallell), og under trening vil vi oppdatere
modellen etter hver lille gruppe. En slik gruppe kalles en **batch**.

`DataLoader` samler bilder og klasser fra datasettet i batches.

`batch_size=64` gir opptil* 64 bilder, og `shuffle=True`
stokker rekkefølgen for hver gjennomgang.


Datasettet inneholder eksemplene; lasteren bestemmer hvordan de grupperes.
`next(iter(...))` henter én gruppe som vi kan undersøke.

*Den siste batchen kan inneholde færre dersom len(train_data) $\bmod$ batch_size $\neq$ 0


In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
images, labels = next(iter(train_loader))
print("Bilder:", images.shape, images.dtype)
print("Etiketter:", labels.shape, labels.dtype)


Bilder: torch.Size([64, 1, 28, 28]) torch.float32
Etiketter: torch.Size([64]) torch.int64


### Refleksjon

Forklar hvert tall i bildegruppens form. Hvorfor har klassene bare
én dimensjon? Hva ville endret seg om `batch_size` var 16?

Skriv svaret ditt i Markdown-cellen under.


**Mitt svar:**

`[64, 1, 28, 28]` betyr 64 bilder, én gråtonekanal, 28 rader og 28 kolonner.
Klassene har formen `[64]` fordi hvert bilde har én riktig klasse.
Med gruppestørrelse 16 ville bildene hatt formen `[16, 1, 28, 28]`
og etikettene `[16]`. Pikselverdiene og klassene er uendret.


## Samme beregning, flere innganger og utganger

Vi bruker fortsatt `z = x @ W + b`. Nå inneholder hver rad i `x` de 784
pikselverdiene fra ett bilde. Hver kolonne i `W` inneholder vektene til én av
de ti klassene. `b` inneholder ett konstantledd per klasse.

Formene blir:

`[B, 784] @ [784, 10] + [10] → [B, 10]`

`B` er antall bilder i batchen. Resultatet inneholder ti scores per bilde.
Vektene er foreløpig tilfeldige, så modellen har ennå ikke lært å gjenkjenne klassene.


## Oppgave 2: Lag ti scores for Fashion-MNIST

Hent `images, labels` fra `train_loader`. Lag `flat_images` med én rad
per bilde. Lag tilfeldige `image_weights` med formen `[784, 10]`, gang dem med
0.01, og lag `image_bias` som ti nuller. Beregn `image_scores`.
Bruk `argmax(dim=1)` til å lage `predictions`. Skriv ut alle formene.

<details>
<summary>Hint</summary>

`torch.randn` lager tilfeldige tall. `argmax(dim=1)` finner kolonnen med høyest score i hver rad.

</details>


In [8]:
images, labels = next(iter(train_loader))
flat_images = images.flatten(start_dim=1)
image_weights = torch.randn(784, 10) * 0.01
image_bias = torch.zeros(10)
image_scores = flat_images @ image_weights + image_bias
predictions = image_scores.argmax(dim=1)
print("Bilder:", images.shape, "Flate bilder:", flat_images.shape)
print("Scores:", image_scores.shape, "Prediksjoner:", predictions.shape)
print("Riktige i gruppen:", (predictions == labels).sum().item())


Bilder: torch.Size([64, 1, 28, 28]) Flate bilder: torch.Size([64, 784])
Scores: torch.Size([64, 10]) Prediksjoner: torch.Size([64])
Riktige i gruppen: 2


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [9]:
assert flat_images.shape == (images.shape[0], 784)
assert image_scores.shape == (images.shape[0], 10)
assert predictions.shape == labels.shape


## Eksempel: la `nn.Linear` håndtere parametrene

`nn.Linear(3, 2)` lærer tre vekter og ett konstantledd per utgang.
Den lagrer vektene som `[utganger, inputs]`, motsatt av vår håndlagde
matrise. Laget utfører den nødvendige transponeringen selv.
`nn.Sequential` kjører flere lag etter hverandre.

Med vår matrisenotasjon er dette `X @ layer.weight.T + layer.bias`.
Antall parametere er `inputs * utganger + utganger`.


In [10]:
small_layer = nn.Linear(3, 2)
print("Resultat:", small_layer(torch.ones(4, 3)).shape)
print("Vekter:", small_layer.weight.shape, "Konstantledd:", small_layer.bias.shape)


Resultat: torch.Size([4, 2])
Vekter: torch.Size([2, 3]) Konstantledd: torch.Size([2])


## Oppgave 3: Bygg en lineær bildemodell

Lag `model = nn.Sequential(...)` med `nn.Flatten()` og et passende `nn.Linear`.
Modellen skal motta bildene direkte og gi ti scores per bilde.
Beregn `model_scores`, og tell parametrene med
`sum(parameter.numel() for parameter in model.parameters())`.

<details>
<summary>Hint</summary>

Hvert bilde har `28 * 28 = 784` piksler. Det er ti mulige klasser.

</details>


In [11]:
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
model_scores = model(images)
num_parameters = sum(parameter.numel() for parameter in model.parameters())
print(model)
print("Utdata:", model_scores.shape, "Parametere:", num_parameters)


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)
Utdata: torch.Size([64, 10]) Parametere: 7850


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [12]:
assert model_scores.shape == (len(images), 10)
assert sum(p.numel() for p in model.parameters()) == 7850


### Refleksjon

Skriv formene gjennom modellen for en gruppe på 32 Fashion-MNIST-bilder.
Hvorfor har modellen 7 850 parametere? Hvorfor forventer vi ikke gode
prediksjoner før trening? Er de ti scorene sannsynligheter?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

`[32, 1, 28, 28] → [32, 784] → [32, 10]`.
Det er 784 vekter og ett konstantledd for hver av ti klasser: `784*10 + 10`.
Parametrene er tilfeldig initialisert, ikke tilpasset bildene.
Scorene er rå tall, ikke sannsynligheter; de kan være negative og summerer ikke til 1.


## Referanser


- [PyTorch – Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).
- [3Blue1Brown – But what is a Neural Network?](https://www.3blue1brown.com/lessons/neural-networks/) - Visuell introduksjon til nevrale nettverk.
